# H-001 · Does OBV-Confirmed Momentum Beat Raw Momentum?

Factor test for **H-001** (equities): whether momentum filtered by On-Balance Volume agreement predicts better than price momentum alone.

- **Idea** — Keep (or weight) price momentum only when OBV trend agrees with price direction; unconfirmed moves are flipped, zeroed, or soft-weighted.
- **Claim** — OBV-confirmed momentum has stronger next-day / next-week predictive power than raw momentum on the same universe.
- **Why it might work** — Volume-backed moves suggest informed flow; unconfirmed price moves are more likely to fade.
- **Data** — Daily OHLCV long panel; compare marginal lift over raw momentum on the same rebalance schedule.

**`normalize` (default `True`):** when `normalize=True`, `add_obv_confirmed_momentum` stores a **cross-sectional (CS)** feature — the percentile rank of the combined signal *within each date*. That is GBM-ready and comparable across names. Set `normalize=False` to keep the raw combined signal in its original scale.

**`mode` options** (column `obv_mom_{mode}`):

| Mode | Behavior |
|------|----------|
| `signed` (default) | Keep momentum when price and OBV trend signs agree; **flip the sign** (same magnitude) when they disagree. |
| `strict_zero` | Keep momentum when signs agree; set to **0** when they disagree. |
| `soft` | Continuous weight: `momentum × soft_weight`, where `soft_weight` is typically `2 × CS pct-rank(OBV trend) − 1` (not a hard agree/disagree gate). |

Use `data.processing.feature_store.add_obv_confirmed_momentum` rather than reimplementing the factor inline.

## 0. Imports & Config

Resolve the repo root, configure paths / dates / factor params (`lookback`, `skip`, `obv_window`, `mode`, `normalize`), and import project helpers (`fetch_top_n_equities`, `add_obv_confirmed_momentum`, Alphalens).

## 1. Data Loading

Load a daily OHLCV long panel for a point-in-time equity universe (same panel for raw and OBV-confirmed signals).

- Prefer project fetchers (`fetch_top_n_equities` / PIT S&P helpers) — no ad-hoc vendor downloads.
- Confirm columns needed by the feature store: `date`, `ticker`, `close`, `volume`.

## 2. Data Cleaning & Engineering

Clean the long panel as needed (missing bars, winsorize, etc.) via project cleaners. Stay in **long format**.

Point-in-time only: features at `t` use information available at or before `t`.

## 3. Modeling / Signal Construction

Build both factors on the **same** cleaned panel so the comparison is fair.

### 3.1 Raw momentum (baseline)

Skip-month style: `P_{t-S} / P_{t-L} - 1` (e.g. L = 252, S = 21). Control factor for the paired test.

### 3.2 OBV-confirmed momentum (H-001)

Call `add_obv_confirmed_momentum` with the chosen `mode` and `normalize`.

- With `normalize=True`, output is a **CS-ranked** feature (`obv_mom_{mode}`).
- With `normalize=False`, output is the unranked combined signal.
- Optionally run more than one `mode` (`signed` / `strict_zero` / `soft`) side by side for sensitivity — note which is primary.

## 4. Evaluation

Paired comparison of **raw momentum** vs **OBV-confirmed momentum** on identical universe, dates, and rebalance schedule.